In [1]:
import sys
path_data = '/home/dal387552/work/Sleep_staging/fitbit'
path_code = '/home/dal387552/work/SS_codes/sleep_staging'
sys.path.insert(0, path_code)

In [2]:
from pre_processing.fitbit_preprocessing import  *

sleep_paths = [
    f"{path_data}/sleep-data/0a85fd46-fada-434c-9f7a-08b81f9ed8e7.csv.gz"
]

heart_paths = [
    f"{path_data}/heart-rate/0a85fd46-fada-434c-9f7a-08b81f9ed8e7.csv.gz",
]

pairs = collect_patient_file_pairs_from_lists(sleep_paths=sleep_paths, heart_paths=heart_paths)
print(pairs)

windowed_dataset = prepare_fitbit_training_data_from_file_lists(
    sleep_paths=sleep_paths,
    heart_paths=heart_paths,
    window_sec=128,
    epoch_sec=30,
    sampling_hz=2.0,
    centered=True,
    normalize_per_night=True,
    normalize_per_window=False,
)

[('0a85fd46-fada-434c-9f7a-08b81f9ed8e7', PosixPath('/home/dal387552/work/Sleep_staging/fitbit/sleep-data/0a85fd46-fada-434c-9f7a-08b81f9ed8e7.csv.gz'), PosixPath('/home/dal387552/work/Sleep_staging/fitbit/heart-rate/0a85fd46-fada-434c-9f7a-08b81f9ed8e7.csv.gz'))]


In [ ]:
from pre_processing.fitbit_preprocessing import prepare_fitbit_training_data_from_directories
from training.data import build_paper_night_sequences, split_by_patient_ids
from modeling import PaperSleepStageModel, PaperSleepStageModelConfig
from training import PaperTrainingConfig, fit_model, evaluate_model

# windowed_dataset = prepare_fitbit_training_data_from_directories(
#     sleep_dir="/Users/raghvendraomer/Downloads/fitbit/sleep-data",
#     heart_dir="/Users/raghvendraomer/Downloads/fitbit/heart-rate",
#     window_sec=128,
#     epoch_sec=30,
#     sampling_hz=2.0,
#     centered=True,
#     normalize_per_night=True,
# )

nightly_dataset = build_paper_night_sequences(
    dataset=windowed_dataset,
    max_epochs=1200,
    pad_value=0.0,
    pad_label=-100,
)

splits = split_by_patient_ids(
    night_dataset=nightly_dataset,
    train_patient_ids=["id_1", "id_2", "id_3"],
    val_patient_ids=["id_4"],
    test_patient_ids=["id_5"],
)

model = PaperSleepStageModel(PaperSleepStageModelConfig())
config = PaperTrainingConfig(
    batch_size=2,
    learning_rate=1e-4,
    weight_decay=0.25,
    epochs=20,
    device="cuda",
)

fit_output = fit_model(
    model=model,
    train_dataset=splits["train"],
    val_dataset=splits["val"],
    config=config,
)

test_metrics = evaluate_model(
    model=fit_output["model"],
    dataset=splits["test"],
    device=config.device,
    batch_size=config.batch_size,
)

print(test_metrics["accuracy"])
print(test_metrics["kappa"])
print(test_metrics["macro_f1"])
print(test_metrics["confusion_matrix"])

In [ ]:
from pre_processing.fitbit_preprocessing import prepare_fitbit_training_data_from_directories
from training.data import build_paper_night_sequences, split_by_patient_ids
from modeling import PaperSleepStageModel, PaperSleepStageModelConfig
from training import PaperTrainingConfig, fit_model, evaluate_model

windowed = prepare_fitbit_training_data_from_directories(
    sleep_dir="/Users/raghvendraomer/Downloads/fitbit/sleep-data",
    heart_dir="/Users/raghvendraomer/Downloads/fitbit/heart-rate",
    window_sec=128,
    epoch_sec=30,
    sampling_hz=2.0,
    centered=True,
    normalize_per_night=True,
)

nightly = build_paper_night_sequences(windowed, max_epochs=1200)

splits = split_by_patient_ids(
    nightly,
    train_patient_ids=["id1", "id2"],
    val_patient_ids=["id3"],
    test_patient_ids=["id4"],
)

model = PaperSleepStageModel(PaperSleepStageModelConfig())
train_cfg = PaperTrainingConfig(batch_size=2, learning_rate=1e-4, weight_decay=0.25, epochs=20, device="cpu")

fit_out = fit_model(model, splits["train"], splits["val"], train_cfg)
test_metrics = evaluate_model(fit_out["model"], splits["test"], device=train_cfg.device)

print(test_metrics["accuracy"], test_metrics["kappa"])
